# Week 7 – Gold Tables | BingeMetrics

**Team:** Team 04  
**Project:** BingeMetrics – OTT & Music Engagement Analytics  
**Notebook:** `notebooks/07_gold_tables.ipynb`

This notebook builds **7 Gold tables** from the DQ-approved Trusted Silver outputs.

### Gold tables created
1. `gold_users`
2. `gold_content`
3. `gold_subscriptions`
4. `gold_sessions`
5. `gold_user_engagement`
6. `gold_content_performance`
7. `gold_daily_engagement`

The Gold layer is designed for analytics and Power BI/dashboard consumption. Quarantine data is not used.


## 1. Gold layer flow

**Trusted Silver → Gold**

- `trusted_users` → `gold_users`
- `trusted_content` → `gold_content`
- `trusted_subscriptions` → `gold_subscriptions`
- `trusted_sessions` + trusted reference tables → `gold_sessions`
- `gold_sessions` → `gold_user_engagement`
- `gold_sessions` + `gold_content` → `gold_content_performance`
- `gold_sessions` → `gold_daily_engagement`

The four detailed Gold tables preserve the trusted records, while the three aggregate Gold tables provide business-ready engagement metrics.


In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA default;

SELECT current_catalog() AS active_catalog,
       current_schema() AS active_schema;


active_catalog,active_schema
workspace,default


In [0]:
# Configuration
CATALOG = "workspace.default"

TRUSTED = {
    "users": f"{CATALOG}.trusted_users",
    "content": f"{CATALOG}.trusted_content",
    "subscriptions": f"{CATALOG}.trusted_subscriptions",
    "sessions": f"{CATALOG}.trusted_sessions"
}

GOLD = {
    "users": f"{CATALOG}.gold_users",
    "content": f"{CATALOG}.gold_content",
    "subscriptions": f"{CATALOG}.gold_subscriptions",
    "sessions": f"{CATALOG}.gold_sessions",
    "user_engagement": f"{CATALOG}.gold_user_engagement",
    "content_performance": f"{CATALOG}.gold_content_performance",
    "daily_engagement": f"{CATALOG}.gold_daily_engagement"
}

print("Trusted inputs:")
for k, v in TRUSTED.items():
    print(k, "->", v)

print("\nGold outputs:")
for k, v in GOLD.items():
    print(k, "->", v)


Trusted inputs:
users -> workspace.default.trusted_users
content -> workspace.default.trusted_content
subscriptions -> workspace.default.trusted_subscriptions
sessions -> workspace.default.trusted_sessions

Gold outputs:
users -> workspace.default.gold_users
content -> workspace.default.gold_content
subscriptions -> workspace.default.gold_subscriptions
sessions -> workspace.default.gold_sessions
user_engagement -> workspace.default.gold_user_engagement
content_performance -> workspace.default.gold_content_performance
daily_engagement -> workspace.default.gold_daily_engagement


## 2. Check Trusted Silver inputs

Only DQ-approved Trusted tables are used as Gold inputs.


In [0]:
%sql
SELECT 'trusted_users' AS table_name, COUNT(*) AS row_count
FROM trusted_users

UNION ALL

SELECT 'trusted_content', COUNT(*)
FROM trusted_content

UNION ALL

SELECT 'trusted_subscriptions', COUNT(*)
FROM trusted_subscriptions

UNION ALL

SELECT 'trusted_sessions', COUNT(*)
FROM trusted_sessions

ORDER BY table_name;


table_name,row_count
trusted_content,2980
trusted_sessions,249701
trusted_subscriptions,34755
trusted_users,24975


## 3. Gold Table 1 – `gold_users`

A clean analytics-friendly user dimension containing the trusted user attributes.


In [0]:
%sql
CREATE OR REPLACE TABLE gold_users
USING DELTA
AS
SELECT
    user_id,
    signup_date,
    signup_cohort,
    age_band,
    region_band,
    preferred_device_segment,
    acquisition_channel,
    user_status
FROM trusted_users
WHERE dq_status = 'PASS';

SELECT COUNT(*) AS rows_created
FROM gold_users;

SELECT *
FROM gold_users
LIMIT 10;


user_id,signup_date,signup_cohort,age_band,region_band,preferred_device_segment,acquisition_channel,user_status
U000001,2025-03-01,2025-Q1,25-34,South-Urban,TV-first,Partner,ACTIVE
U000002,2024-05-22,2024-Q2,18-24,East-Urban,TV-first,Campaign,ACTIVE
U000003,2025-05-01,2025-Q2,35-44,East-Urban,Mobile-first,Organic,ACTIVE
U000004,2024-08-16,2024-Q3,18-24,North-Urban,Mobile-first,Organic,ACTIVE
U000005,2024-10-26,2024-Q4,55+,North-Urban,TV-first,Campaign,ACTIVE
U000006,2025-04-02,2025-Q2,25-34,West-Urban,Mobile-first,Organic,ACTIVE
U000007,2025-05-23,2025-Q2,25-34,West-Urban,Multi-device,Organic,ACTIVE
U000008,2025-03-28,2025-Q1,25-34,West-Urban,TV-first,Organic,ACTIVE
U000009,2024-02-21,2024-Q1,25-34,North-Urban,Multi-device,Partner,DORMANT
U000010,2025-04-14,2025-Q2,25-34,North-Urban,Desktop-first,Partner,ACTIVE


## 4. Gold Table 2 – `gold_content`

A clean content dimension used for content-level analytics.


In [0]:
%sql
CREATE OR REPLACE TABLE gold_content
USING DELTA
AS
SELECT
    content_id,
    content_type,
    title_label,
    genre,
    language,
    duration_seconds,
    is_platform_original,
    maturity_band,
    release_year,
    series_collection_id,
    available_from_date,
    available_to_date,
    catalog_status
FROM trusted_content
WHERE dq_status = 'PASS';

SELECT COUNT(*) AS rows_created
FROM gold_content;

SELECT *
FROM gold_content
LIMIT 10;


content_id,content_type,title_label,genre,language,duration_seconds,is_platform_original,maturity_band,release_year,series_collection_id,available_from_date,available_to_date,catalog_status
C000001,VIDEO,Parallel Notes 0001,Sports Stories,Telugu,2526,false,A,2017,null,2017-01-01,null,LIMITED
C000002,MUSIC,Coastal Rhythms 0002,Jazz,Telugu,210,false,U/A 16+,2002,null,2002-01-01,null,AVAILABLE
C000003,MUSIC,Open Voices 0003,Devotional,Hindi,260,false,U,2013,null,2013-01-01,null,AVAILABLE
C000004,MUSIC,Midnight Journeys 0004,Indie Pop,Kannada,166,true,U,2016,null,2016-01-01,null,AVAILABLE
C000005,PODCAST,Neon Archives 0005,Careers,Spanish,4517,false,U/A 16+,2002,null,2002-01-01,null,AVAILABLE
C000006,VIDEO,Silver Horizons 0006,Documentary,Hindi,6503,false,U/A 7+,1999,COL00001,1999-01-01,null,LIMITED
C000007,VIDEO,Urban Stories 0007,Comedy,Hindi,3063,false,U/A 13+,2026,null,2026-01-01,null,AVAILABLE
C000008,VIDEO,Hidden Patterns 0008,Sports Stories,English,2346,true,U/A 13+,1993,null,1993-01-01,null,AVAILABLE
C000009,VIDEO,Solar Frames 0009,Drama,Telugu,5528,false,U,2026,COL00002,2026-01-01,null,AVAILABLE
C000010,VIDEO,Quiet Circuits 0010,Comedy,Spanish,2415,false,A,2012,null,2012-01-01,null,AVAILABLE


## 5. Gold Table 3 – `gold_subscriptions`

A clean subscription table connected to users through `user_id`.


In [0]:
%sql
CREATE OR REPLACE TABLE gold_subscriptions
USING DELTA
AS
SELECT
    subscription_id,
    user_id,
    plan_code,
    billing_cycle,
    period_start_date,
    period_end_date,
    lifecycle_status,
    auto_renew_flag,
    cancellation_reason_group
FROM trusted_subscriptions
WHERE dq_status = 'PASS';

SELECT COUNT(*) AS rows_created
FROM gold_subscriptions;

SELECT *
FROM gold_subscriptions
LIMIT 10;


subscription_id,user_id,plan_code,billing_cycle,period_start_date,period_end_date,lifecycle_status,auto_renew_flag,cancellation_reason_group
S0000001,U000001,Premium,Monthly,2025-04-03,2026-06-30,ACTIVE,true,null
S0000002,U000002,Premium,Annual,2024-07-06,2026-06-30,ACTIVE,true,null
S0000003,U000003,Premium,Monthly,2025-06-15,2026-06-30,ACTIVE,true,null
S0000004,U000004,AudioPlus,Monthly,2024-08-31,2026-06-30,ACTIVE,false,null
S0000005,U000005,Student,Monthly,2024-11-14,2026-06-30,ACTIVE,false,null
S0000006,U000006,AudioPlus,Monthly,2025-05-08,2026-06-30,ACTIVE,true,null
S0000007,U000007,Premium,Monthly,2025-07-03,2026-06-30,ACTIVE,true,null
S0000008,U000008,AudioPlus,Monthly,2025-04-26,2026-06-30,ACTIVE,true,null
S0000009,U000009,Standard,Monthly,2024-03-06,2026-06-30,ACTIVE,false,null
S0000010,U000010,Premium,Monthly,2025-05-08,2026-06-30,ACTIVE,true,null


## 6. Gold Table 4 – `gold_sessions`

An enriched session-level fact table.

It combines trusted session records with user, content and subscription attributes. The join is performed after DQ processing so that only trusted reference data is used.


In [0]:
%sql
CREATE OR REPLACE TABLE gold_sessions
USING DELTA
AS
SELECT
    s.session_id,
    s.user_id,
    s.content_id,
    s.subscription_id,

    CAST(s.session_start_ts AS DATE) AS session_date,
    s.session_start_ts,
    s.session_end_ts,

    s.device_type,
    s.source_app_channel,
    s.source_app_version,

    s.consumed_seconds,
    s.max_playback_position_seconds,
    s.end_reason,
    s.reported_completed_flag,
    s.reported_skipped_flag,

    u.signup_cohort,
    u.age_band,
    u.region_band,
    u.preferred_device_segment,
    u.acquisition_channel,
    u.user_status,

    c.content_type,
    c.genre,
    c.language,
    c.duration_seconds AS content_duration_seconds,
    c.is_platform_original,

    sub.plan_code,
    sub.billing_cycle,
    sub.lifecycle_status AS subscription_status,
    sub.auto_renew_flag

FROM trusted_sessions s

LEFT JOIN gold_users u
    ON s.user_id = u.user_id

LEFT JOIN gold_content c
    ON s.content_id = c.content_id

LEFT JOIN gold_subscriptions sub
    ON s.subscription_id = sub.subscription_id

WHERE s.dq_status = 'PASS';

SELECT COUNT(*) AS rows_created
FROM gold_sessions;

SELECT *
FROM gold_sessions
LIMIT 10;


session_id,user_id,content_id,subscription_id,session_date,session_start_ts,session_end_ts,device_type,source_app_channel,source_app_version,consumed_seconds,max_playback_position_seconds,end_reason,reported_completed_flag,reported_skipped_flag,signup_cohort,age_band,region_band,preferred_device_segment,acquisition_channel,user_status,content_type,genre,language,content_duration_seconds,is_platform_original,plan_code,billing_cycle,subscription_status,auto_renew_flag
EGS000000003,U018708,C000580,S0018708,2026-02-22,2026-02-22T04:54:14.000,2026-02-22T05:50:47.000,Mobile,tv_app,7.3,3368,3368,completed,true,false,2024-Q2,55+,East-Urban,Mobile-first,Organic,ACTIVE,PODCAST,Culture,Tamil,3460,false,Standard,Monthly,ACTIVE,true
EGS000000005,U011749,C002597,S0011749,2026-03-11,2026-03-11T13:29:32.000,2026-03-11T13:34:00.000,Mobile,tv_app,7.3,104,104,user_stopped,false,false,2025-Q1,55+,South-Urban,Mobile-first,Organic,ACTIVE,MUSIC,Folk Fusion,English,165,false,AudioPlus,Monthly,ACTIVE,true
EGS000000016,U005348,C001997,S0005348,2026-01-28,2026-01-28T02:23:47.000,2026-01-28T03:37:53.000,Mobile,web_player,6.5,4309,4309,app_closed,false,false,2024-Q3,25-34,East-Urban,Mobile-first,Organic,ACTIVE,VIDEO,Sports Stories,English,6165,true,Standard,Quarterly,ACTIVE,true
EGS000000031,U007892,C002035,S0007892,2026-02-14,2026-02-14T23:09:34.000,2026-02-15T00:09:35.000,Mobile,mobile_app,8.5,3593,3593,completed,true,false,2024-Q1,25-34,East-Urban,Mobile-first,Referral,DORMANT,VIDEO,Thriller,Kannada,3923,false,Basic,Monthly,ACTIVE,true
EGS000000035,U019983,C002685,S0019983,2026-03-12,2026-03-12T03:32:47.000,2026-03-12T03:38:00.000,Web,tv_app,6.5,203,203,user_stopped,false,false,2025-Q3,25-34,North-Urban,Multi-device,Partner,ACTIVE,MUSIC,Electronic,English,248,false,Premium,Quarterly,ACTIVE,true
EGS000000046,U020306,C002645,S0020306,2026-01-30,2026-01-30T06:57:48.000,2026-01-30T07:04:16.000,Connected TV,tablet_app,5.5,269,269,user_stopped,false,false,2025-Q1,18-24,North-Urban,TV-first,Organic,ACTIVE,MUSIC,Instrumental,English,331,true,Student,Monthly,ACTIVE,true
EGS000000052,U021490,C002329,S0021490,2026-01-04,2026-01-04T18:17:48.000,2026-01-04T18:18:44.000,Mobile,web_player,5.9,45,45,skipped,false,true,2024-Q2,18-24,East-Urban,Mobile-first,Organic,ACTIVE,MUSIC,Electronic,Korean,265,false,Standard,Monthly,ACTIVE,true
EGS000000080,U012533,C000973,S0012533,2026-03-14,2026-03-14T18:44:07.000,2026-03-14T18:46:47.000,Mobile,tv_app,8.6,47,47,skipped,false,true,2024-Q4,18-24,South-Semiurban,Mobile-first,Referral,ACTIVE,VIDEO,Learning,Spanish,2149,false,Student,Monthly,ACTIVE,false
EGS000000098,U007771,C001978,S0007771,2026-03-05,2026-03-05T12:20:06.000,2026-03-05T13:27:11.000,Connected TV,tv_app,7.3,3959,3959,user_stopped,false,false,2025-Q4,25-34,Central-Mixed,Desktop-first,Partner,ACTIVE,VIDEO,Learning,Telugu,4743,false,AudioPlus,Monthly,ACTIVE,true
EGS000000103,U019312,C002005,S0019312,2026-03-12,2026-03-12T21:52:38.000,2026-03-12T22:22:21.000,Connected TV,mobile_app,5.5,1723,1723,user_stopped,false,false,2024-Q2,25-34,South-Urban,TV-first,Campaign,ACTIVE,PODCAST,Culture,English,3535,false,Standard,Monthly,ACTIVE,true


## 7. Gold Table 5 – `gold_user_engagement`

User-level engagement summary for dashboard and retention analysis.

Metrics include:
- total sessions
- total consumed seconds
- total completed sessions
- total skipped sessions
- average session duration
- first and last session dates


In [0]:
%sql
CREATE OR REPLACE TABLE gold_user_engagement
USING DELTA
AS
SELECT
    user_id,

    COUNT(*) AS total_sessions,

    SUM(consumed_seconds) AS total_consumed_seconds,

    ROUND(SUM(consumed_seconds) / 60.0, 2)
        AS total_consumed_minutes,

    SUM(CASE
            WHEN reported_completed_flag = true THEN 1
            ELSE 0
        END) AS completed_sessions,

    SUM(CASE
            WHEN reported_skipped_flag = true THEN 1
            ELSE 0
        END) AS skipped_sessions,

    ROUND(AVG(consumed_seconds), 2)
        AS average_session_seconds,

    MIN(session_date) AS first_session_date,
    MAX(session_date) AS last_session_date

FROM gold_sessions
GROUP BY user_id;

SELECT COUNT(*) AS users_created
FROM gold_user_engagement;

SELECT *
FROM gold_user_engagement
ORDER BY total_consumed_seconds DESC
LIMIT 10;


user_id,total_sessions,total_consumed_seconds,total_consumed_minutes,completed_sessions,skipped_sessions,average_session_seconds,first_session_date,last_session_date
U022821,16,55091,918.18,7,1,3443.19,2026-01-27,2026-03-27
U021041,20,54356,905.93,8,5,2717.8,2026-01-03,2026-03-30
U011808,24,54134,902.23,9,7,2255.58,2026-01-01,2026-03-31
U014810,18,49149,819.15,7,3,2730.5,2026-01-01,2026-03-30
U004095,23,47605,793.42,10,5,2069.78,2026-01-04,2026-03-31
U011993,18,47592,793.20,9,3,2644.0,2026-01-03,2026-03-27
U001192,18,46623,777.05,9,1,2590.17,2026-01-03,2026-03-30
U014347,18,46495,774.92,9,2,2583.06,2026-01-03,2026-03-30
U009112,17,46490,774.83,13,0,2734.71,2026-01-05,2026-03-24
U014956,16,46476,774.60,11,1,2904.75,2026-01-14,2026-03-28


## 8. Gold Table 6 – `gold_content_performance`

Content-level performance summary.

This table helps identify the most watched content, most completed content and content generating the highest engagement time.


In [0]:
%sql
CREATE OR REPLACE TABLE gold_content_performance
USING DELTA
AS
SELECT
    content_id,

    MAX(content_type) AS content_type,
    MAX(genre) AS genre,
    MAX(language) AS language,
    MAX(is_platform_original) AS is_platform_original,

    COUNT(*) AS total_sessions,
    COUNT(DISTINCT user_id) AS unique_viewers,

    SUM(consumed_seconds) AS total_consumed_seconds,

    ROUND(SUM(consumed_seconds) / 60.0, 2)
        AS total_consumed_minutes,

    ROUND(AVG(consumed_seconds), 2)
        AS average_session_seconds,

    SUM(CASE
            WHEN reported_completed_flag = true THEN 1
            ELSE 0
        END) AS completed_sessions,

    SUM(CASE
            WHEN reported_skipped_flag = true THEN 1
            ELSE 0
        END) AS skipped_sessions,

    ROUND(
        100.0 * SUM(
            CASE
                WHEN reported_completed_flag = true THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS completion_rate_percent

FROM gold_sessions
WHERE content_id IS NOT NULL
GROUP BY content_id;

SELECT COUNT(*) AS content_rows_created
FROM gold_content_performance;

SELECT *
FROM gold_content_performance
ORDER BY total_consumed_seconds DESC
LIMIT 10;


content_id,content_type,genre,language,is_platform_original,total_sessions,unique_viewers,total_consumed_seconds,total_consumed_minutes,average_session_seconds,completed_sessions,skipped_sessions,completion_rate_percent
C000854,VIDEO,Learning,English,true,113,113,511449,8524.15,4526.1,50,21,44.25
C001682,VIDEO,Thriller,English,true,106,106,470387,7839.78,4437.61,50,21,47.17
C001872,VIDEO,Animation,Tamil,false,102,101,460370,7672.83,4513.43,46,16,45.10
C001060,VIDEO,Comedy,English,true,96,96,452458,7540.97,4713.1,44,16,45.83
C001584,VIDEO,Documentary,Kannada,false,92,90,444713,7411.88,4833.84,50,13,54.35
C002100,VIDEO,Comedy,English,false,106,105,441987,7366.45,4169.69,38,14,35.85
C000510,VIDEO,Drama,English,true,92,92,439077,7317.95,4772.58,44,12,47.83
C001958,VIDEO,Learning,Hindi,false,101,101,437843,7297.38,4335.08,38,19,37.62
C000169,VIDEO,Drama,Spanish,false,98,98,435622,7260.37,4445.12,39,17,39.80
C000369,VIDEO,Drama,Korean,false,94,94,433790,7229.83,4614.79,32,11,34.04


## 9. Gold Table 7 – `gold_daily_engagement`

Daily platform engagement summary.

This table is suitable for time-series dashboards and trend analysis.


In [0]:
%sql
CREATE OR REPLACE TABLE gold_daily_engagement
USING DELTA
AS
SELECT
    session_date,

    COUNT(*) AS total_sessions,
    COUNT(DISTINCT user_id) AS unique_users,
    COUNT(DISTINCT content_id) AS unique_content_items,

    SUM(consumed_seconds) AS total_consumed_seconds,

    ROUND(SUM(consumed_seconds) / 60.0, 2)
        AS total_consumed_minutes,

    ROUND(AVG(consumed_seconds), 2)
        AS average_session_seconds,

    SUM(CASE
            WHEN reported_completed_flag = true THEN 1
            ELSE 0
        END) AS completed_sessions,

    SUM(CASE
            WHEN reported_skipped_flag = true THEN 1
            ELSE 0
        END) AS skipped_sessions,

    ROUND(
        100.0 * SUM(
            CASE
                WHEN reported_completed_flag = true THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS completion_rate_percent

FROM gold_sessions
GROUP BY session_date;

SELECT COUNT(*) AS daily_rows_created
FROM gold_daily_engagement;

SELECT *
FROM gold_daily_engagement
ORDER BY session_date
LIMIT 20;


session_date,total_sessions,unique_users,unique_content_items,total_consumed_seconds,total_consumed_minutes,average_session_seconds,completed_sessions,skipped_sessions,completion_rate_percent
2025-12-20,500,492,465,780396,13006.60,1560.79,183,105,36.60
2026-01-01,2738,2586,1812,4125957,68765.95,1506.92,990,576,36.16
2026-01-02,2774,2633,1777,4167149,69452.48,1502.22,1072,538,38.64
2026-01-03,2790,2639,1791,4117139,68618.98,1475.68,1064,532,38.14
2026-01-04,2723,2571,1771,4034437,67240.62,1481.61,1008,520,37.02
2026-01-05,2693,2552,1767,4051622,67527.03,1504.5,1025,533,38.06
2026-01-06,2753,2593,1768,4167139,69452.32,1513.67,1061,507,38.54
2026-01-07,2852,2687,1818,4296223,71603.72,1506.39,1080,580,37.87
2026-01-08,2866,2712,1813,4206845,70114.08,1467.85,1080,560,37.68
2026-01-09,2800,2647,1820,4030967,67182.78,1439.63,1058,564,37.79


## 10. Validate all 7 Gold tables

This section confirms that all seven Gold tables were created and reports their row counts.


In [0]:
%sql
SELECT 'gold_users' AS table_name, COUNT(*) AS row_count
FROM gold_users

UNION ALL

SELECT 'gold_content', COUNT(*)
FROM gold_content

UNION ALL

SELECT 'gold_subscriptions', COUNT(*)
FROM gold_subscriptions

UNION ALL

SELECT 'gold_sessions', COUNT(*)
FROM gold_sessions

UNION ALL

SELECT 'gold_user_engagement', COUNT(*)
FROM gold_user_engagement

UNION ALL

SELECT 'gold_content_performance', COUNT(*)
FROM gold_content_performance

UNION ALL

SELECT 'gold_daily_engagement', COUNT(*)
FROM gold_daily_engagement

ORDER BY table_name;


table_name,row_count
gold_content,2980
gold_content_performance,3400
gold_daily_engagement,91
gold_sessions,249701
gold_subscriptions,34755
gold_user_engagement,24999
gold_users,24975


## 11. Basic Gold-layer quality checks

The checks below verify that important Gold keys are not null and that the session enrichment did not multiply the trusted session rows.


In [0]:
%sql
SELECT
    'gold_users' AS table_name,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT user_id) AS distinct_keys,
    SUM(CASE WHEN user_id IS NULL THEN 1 ELSE 0 END) AS null_keys
FROM gold_users

UNION ALL

SELECT
    'gold_content',
    COUNT(*),
    COUNT(DISTINCT content_id),
    SUM(CASE WHEN content_id IS NULL THEN 1 ELSE 0 END)
FROM gold_content

UNION ALL

SELECT
    'gold_subscriptions',
    COUNT(*),
    COUNT(DISTINCT subscription_id),
    SUM(CASE WHEN subscription_id IS NULL THEN 1 ELSE 0 END)
FROM gold_subscriptions

UNION ALL

SELECT
    'gold_sessions',
    COUNT(*),
    COUNT(DISTINCT session_id),
    SUM(CASE WHEN session_id IS NULL THEN 1 ELSE 0 END)
FROM gold_sessions;


table_name,total_rows,distinct_keys,null_keys
gold_users,24975,24975,0
gold_content,2980,2980,0
gold_subscriptions,34755,34755,0
gold_sessions,249701,249700,1


In [0]:
# Session join multiplication guard
trusted_session_count = spark.table(TRUSTED["sessions"]).filter("dq_status = 'PASS'").count()
gold_session_count = spark.table(GOLD["sessions"]).count()

print("Trusted session rows :", trusted_session_count)
print("Gold session rows    :", gold_session_count)

if trusted_session_count != gold_session_count:
    raise ValueError(
        "Gold session join multiplication guard failed: "
        "gold_sessions row count differs from trusted_sessions."
    )

print("PASS: gold_sessions preserves the trusted physical session row count.")


Trusted session rows : 249701
Gold session rows    : 249701
PASS: gold_sessions preserves the trusted physical session row count.


## 12. Gold table summary

| # | Gold table | Grain | Main purpose |
|---|---|---|---|
| 1 | `gold_users` | One row per user | User dimension |
| 2 | `gold_content` | One row per content item | Content dimension |
| 3 | `gold_subscriptions` | One row per subscription | Subscription analytics |
| 4 | `gold_sessions` | One row per session | Enriched session fact |
| 5 | `gold_user_engagement` | One row per user | User engagement KPIs |
| 6 | `gold_content_performance` | One row per content item | Content performance KPIs |
| 7 | `gold_daily_engagement` | One row per day | Daily engagement trends |

### Final architecture

**Silver → DQ Trusted → Gold**

`trusted_users` → `gold_users` → `gold_user_engagement`

`trusted_content` → `gold_content` → `gold_content_performance`

`trusted_subscriptions` → `gold_subscriptions`

`trusted_sessions` → `gold_sessions` → `gold_user_engagement`

`gold_sessions` → `gold_daily_engagement`
